# 05 — Gradio Demo

**用途**：在 Colab 啟動與本機相同的四狀態介面，分析流程為：

1. 選擇短片。
2. 依序執行影片解碼、Pose extraction、Event detection 與 Video annotation。
3. 在同一工作區檢視標註影片、事件區間、Track ID、實際 `Rules fired` 與
   `events.json`。無事件時會顯示明確的 0 事件結論。

本機使用者可直接執行 README 的 `uv run python -m fall_detection.app.gradio_app
--no-share`，不需要先操作本 notebook。本 notebook 保留作為 Colab 重現路徑。

**執行方式**：`Runtime → Run all`（GPU runtime，T4 即可；CPU 也能執行但較慢）。
最後一格會持續執行，因為 `gr.Blocks.launch()` 正在提供網頁服務；停止時手動中斷該格。

範例使用 `fall-06` 與 `adl-01`，分別對應 README 中的實際 ALARM 與 0 事件媒體。


In [ ]:
!nvidia-smi


In [ ]:
import os
REPOSITORY_URL = ""  # @param {type:"string"}

if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        if not REPOSITORY_URL:
            raise ValueError('Set REPOSITORY_URL to this repository HTTPS clone URL')
        !git clone -q {REPOSITORY_URL}
    %cd fall-detection-pose
!pip -q install -e ".[infer,demo]" pytest

# 同前幾本 notebook 的坑:pip install -e 之後 site.py 不會自動重新掃描 .pth,
# 這裡直接跑會 ModuleNotFoundError。reload(site) + 手動加 src/ 到 sys.path 雙重保險。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)

In [ ]:
# 規則引擎 + gradio_app 純函式單元測試(純 CPU,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
VIDEO_DIR = f'{DATA_DIR}/videos'

# 與 README 的實際 pipeline 媒體一致：fall-06 形成 ALARM；adl-01 為 0 事件。
# 找不到時保留空範例區，使用者仍可自行上傳影片。
example_candidates = [f'{VIDEO_DIR}/fall-06.mp4', f'{VIDEO_DIR}/adl-01.mp4']
examples = [p for p in example_candidates if os.path.exists(p)]
missing = [p for p in example_candidates if p not in examples]
if missing:
    print(f'警告：找不到範例影片 {missing}（notebook 02 是否已跑過？），仍可手動上傳')
print('範例影片：', examples)


In [ ]:
from fall_detection.app.gradio_app import build_demo

CONFIG_PATH = os.path.abspath('config.yaml')
demo = build_demo(config_path=CONFIG_PATH, example_videos=examples)

print('=== 啟動中,請等下面印出 https://xxxx.gradio.live 連結 ===')
demo.queue().launch(share=True, max_file_size='200mb')

## 結果與 README 媒體

README 的 `assets/demo_fall.gif`、`assets/demo_adl.png` 與 `assets/demo_mobile.png`
由本專案的 Playwright 擷取腳本從實際 pipeline 結果產生。若介面改版，請在本機重新執行
`scripts/capture_demo_media.py`，不要手動拼接推論數值或修改事件結果。
